# Dynamic In-Context Adaptation Under Shifting SLA Priorities

Four orchestrators across 502 lists (251 Scheme B + 251 Scheme C)

1. Llama-B no-adapt
2. Llama-B prompt-adapt (3 Scheme C demos injected at boundary)
3. TOPSIS-stale (Scheme B weights throughout)
4. LambdaMART-B no-retrain

In [1]:
# CELL 1: Install dependencies
!pip install -q transformers accelerate peft
!pip install -U bitsandbytes>=0.46.1
!pip install lightgbm scikit-learn numpy pandas scipy matplotlib -q


In [10]:
# CELL 2: Configuration — edit ORCHESTRATOR to switch between runs
# Options: 'llama_noadapt' | 'llama_promptadapt' | 'topsis_stale' | 'lambdamart_b'
# Options: 'qwen_noadapt' | 'qwen_promptadapt' | 'topsis_stale' | 'lambdamart_b'

ORCHESTRATOR = 'qwen_promptadapt'

MODEL_NAME      = 'Qwen2.5-7B'  # 'Qwen2.5-7B' , 'Llama3.1-8B'
HF_TOKEN        = ''
ADAPTER_B_HF_ID = 'NajatAlsa/qwen-scheme-b'
HF_MODEL_ID     = 'Qwen/Qwen2.5-7B-Instruct'   # 'Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct'

WEIGHTS_B = {
    'Response_Time_norm': 0.3,
    'Availability_norm':  0.1,
    'Throughput_norm':    0.1,
    'Reliability_norm':   0.1,
    'Latency_norm':       0.4
}

print(f'Orchestrator: {ORCHESTRATOR}')


Orchestrator: qwen_promptadapt


In [3]:
# CELL 3: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os, sys

BASE_PATH    = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/'
RESULTS_PATH = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_Dynamic_Adaptation/'
os.makedirs(RESULTS_PATH, exist_ok=True)

TEST_B_PATH  = BASE_PATH + 'Scheme_B/obj3_test_B.csv'
TEST_C_PATH  = BASE_PATH + 'Scheme_C/obj3_test_C.csv'
TRAIN_B_PATH = BASE_PATH + 'Scheme_B/obj3_train_B.csv'
VAL_B_PATH   = BASE_PATH + 'Scheme_B/obj3_val_B.csv'
VAL_C_PATH  = BASE_PATH + 'Scheme_C/obj3_val_C.csv'

sys.path.append('/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/')
from prompt_config import SYSTEM_INSTRUCTION, build_user_content, build_assistant_output

print(f'Results: {RESULTS_PATH}')


Mounted at /content/drive
Results: /content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_Dynamic_Adaptation/


In [4]:
# CELL 4: Imports
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import numpy as np
import torch
import re
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


Device: cuda


In [5]:
# CELL 5: Load data and build trace

import pandas as pd

test_b = pd.read_csv(TEST_B_PATH)   # Phase 1 evaluation
test_c = pd.read_csv(TEST_C_PATH)   # Phase 2 evaluation
val_c  = pd.read_csv(VAL_C_PATH)    # demos only

list_ids_b      = test_b['list_id'].unique().tolist()   # 251 Phase 1 lists
list_ids_c      = test_c['list_id'].unique().tolist()   # 251 Phase 2 lists
EVAL_LIST_IDS_C = list_ids_c

# 5 distinctive Scheme C demos from VAL set
# Selected: top-1 node has very high RT but wins due to high Availability

DEMO_LIST_IDS_C = [14, 51, 97, 200, 217, 36, 91]
demo_groups_c = [val_c[val_c['list_id'] == lid].copy() for lid in DEMO_LIST_IDS_C]


print(f'Phase 1 (Scheme B): {len(list_ids_b)} test lists')
print(f'Phase 2 (Scheme C): {len(EVAL_LIST_IDS_C)} test lists')
print(f'Demos (from val_c): {DEMO_LIST_IDS_C}')

Phase 1 (Scheme B): 251 test lists
Phase 2 (Scheme C): 251 test lists
Demos (from val_c): [14, 51, 97, 200, 217, 36, 91]


In [6]:
# CELL 6: Shared metric functions

NORM_COLS = ['Response_Time_norm', 'Availability_norm', 'Throughput_norm',
             'Reliability_norm', 'Latency_norm']

def compute_metrics_df(group_df, pred_ranks_dict):
    display_ids = group_df['display_id'].tolist()
    true_ranks  = dict(zip(group_df['display_id'], group_df['list_rank']))
    pred_list   = [pred_ranks_dict[did] for did in display_ids]
    true_list   = [true_ranks[did] for did in display_ids]
    mae         = float(np.mean(np.abs(np.array(pred_list) - np.array(true_list))))
    spearman, _ = spearmanr(pred_list, true_list)
    spearman    = float(spearman) if not np.isnan(spearman) else 0.0
    y_true      = np.array([[11 - r for r in true_list]])
    y_pred      = np.array([[11 - r for r in pred_list]])
    ndcg        = float(ndcg_score(y_true, y_pred))
    true_top1   = group_df[group_df['list_rank'] == 1]['display_id'].values[0]
    pred_top1   = min(pred_ranks_dict, key=pred_ranks_dict.get)
    top1        = int(true_top1 == pred_top1)
    return {'mae': mae, 'spearman': spearman, 'ndcg': ndcg, 'top1': top1}

print('Metrics defined.')


Metrics defined.


In [7]:
# CELL 7a: Load model
# Do NOT re-run just to change inference settings — use Cell 7b instead

if ORCHESTRATOR not in ('topsis_stale', 'lambdamart_b'):
    from peft import PeftModel
    from huggingface_hub import snapshot_download

    print(f'Downloading {MODEL_NAME} adapter...')
    adapter_local = snapshot_download(repo_id=ADAPTER_B_HF_ID, token=HF_TOKEN)

    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_ID, quantization_config=bnb_config,
        device_map='auto', token=HF_TOKEN
    )
    model = PeftModel.from_pretrained(base_model, adapter_local)
    model.eval()
    print(f'{MODEL_NAME} loaded.')

elif ORCHESTRATOR == 'topsis_stale':
    def get_predictions(group_df, **kwargs):
        w        = np.array(list(WEIGHTS_B.values()))
        matrix   = group_df[NORM_COLS].values.astype(float)
        weighted = matrix * w
        ideal_best  = weighted.max(axis=0)
        ideal_worst = weighted.min(axis=0)
        dist_best   = np.sqrt(((weighted - ideal_best)  ** 2).sum(axis=1))
        dist_worst  = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))
        closeness   = dist_worst / (dist_best + dist_worst + 1e-10)
        ranks       = pd.Series(closeness).rank(ascending=False, method='min').astype(int).values
        return dict(zip(group_df['display_id'].tolist(), ranks))
    print('TOPSIS-stale ready.')

elif ORCHESTRATOR == 'lambdamart_b':
    train_b = pd.read_csv(TRAIN_B_PATH)
    val_b   = pd.read_csv(VAL_B_PATH)

    def prep_lgb(df):
        X, y, g = [], [], []
        for lid in df['list_id'].unique():
            grp = df[df['list_id'] == lid]
            X.append(grp[NORM_COLS].values)
            y.append((11 - grp['list_rank']).values)
            g.append(len(grp))
        return np.vstack(X), np.concatenate(y), g

    X_tr, y_tr, g_tr = prep_lgb(train_b)
    X_vl, y_vl, g_vl = prep_lgb(val_b)
    dtrain = lgb.Dataset(X_tr, label=y_tr, group=g_tr)
    dval   = lgb.Dataset(X_vl, label=y_vl, group=g_vl)
    params = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [1,5,10],
        'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1, 'seed': 42,
        'bagging_fraction': 0.9, 'bagging_freq': 5, 'feature_fraction': 0.9,
    }
    lgb_model = lgb.train(
        params, dtrain, num_boost_round=500, valid_sets=[dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)]
    )

    def get_predictions(group_df, **kwargs):
        X      = group_df[NORM_COLS].values.astype(float)
        scores = lgb_model.predict(X)
        ranks  = pd.Series(scores).rank(ascending=False, method='min').astype(int).values
        return dict(zip(group_df['display_id'].tolist(), ranks))
    print('LambdaMART-B trained and ready.')

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B loaded.


In [8]:
# CELL 7b: Define inference functions
# Re-run this cell ONLY to update inference settings (temperature, demos, hints etc.)
# No need to reload the model

if ORCHESTRATOR not in ('topsis_stale', 'lambdamart_b'):

    def build_prompt(group_df, demo_groups=None, scheme_hint=None):
        messages = [{'role': 'system', 'content': SYSTEM_INSTRUCTION}]
        if demo_groups:
            for demo_df in demo_groups:
                messages.append({'role': 'user',      'content': build_user_content(demo_df)})
                messages.append({'role': 'assistant', 'content': build_assistant_output(demo_df)})
        user_content = build_user_content(group_df)
        if scheme_hint:
            user_content = scheme_hint + "\n\n" + user_content
        messages.append({'role': 'user', 'content': user_content})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def run_inference(group_df, demo_groups=None, scheme_hint=None):
        prompt       = build_prompt(group_df, demo_groups, scheme_hint)
        inputs       = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        input_length = inputs['input_ids'].shape[1]
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=200, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    def parse_ranking(generated_text, group_df):
        display_ids = group_df['display_id'].tolist()
        pred_ranks, rank = {}, 1
        for line in generated_text.strip().split('\n'):
            for did in display_ids:
                if re.search(r'\b' + re.escape(did) + r'\b', line) and did not in pred_ranks:
                    pred_ranks[did] = rank; rank += 1; break
        for did in display_ids:
            if did not in pred_ranks:
                pred_ranks[did] = rank; rank += 1
        return pred_ranks

    print(f'Inference functions defined for {MODEL_NAME}. Greedy decoding.')

Inference functions defined for Qwen2.5-7B. Greedy decoding.


In [ ]:
# CELL 8: Run the selected orchestrator across all 502 lists
# Checkpoint-enabled: safe to interrupt and resume
# IMPORTANT: done_ids uses 'phase_listid' key to avoid collision
# (Phase B and Phase C both have list_ids 0-250)

import torch

OUTPUT_FILE = RESULTS_PATH + f'{ORCHESTRATOR}_502.csv'

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_ids = set((existing['phase'] + '_' + existing['list_id'].astype(str)).tolist())
    results  = existing.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_ids)} lists done')
else:
    done_ids = set()
    results  = []

# Build full trace: Phase 1 = Scheme B, Phase 2 = Scheme C
trace = (
    [(lid, test_b, 'B', i)                   for i, lid in enumerate(list_ids_b)] +
    [(lid, test_c, 'C', len(list_ids_b) + i) for i, lid in enumerate(EVAL_LIST_IDS_C)]
)

print(f'Running orchestrator: {ORCHESTRATOR}')
print(f'Total lists to run:   {len(trace)}')
print(f'Phase boundary at:    list index {len(list_ids_b)}')

COT_HINT = (
    "New ranking rule: sort nodes by Availability descending. "
    "Use Reliability as tiebreaker. "
    "Ignore all other metrics. "
    "Now rank:"

)

for list_index, (lid, source_df, phase, idx) in enumerate(trace):
    unique_key = f'{phase}_{lid}'
    if unique_key in done_ids:
        continue

    if list_index % 25 == 0:
        print(f'  [{list_index}/{len(trace)}] Phase {phase}, list_id {lid}...')

    group = source_df[source_df['list_id'] == lid].copy()
    group = group.sample(frac=1, random_state=lid).reset_index(drop=True)

    # Get predictions based on orchestrator type
    if ORCHESTRATOR not in ('topsis_stale', 'lambdamart_b'):
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        if phase == 'C' and 'promptadapt' in ORCHESTRATOR:
            generated = run_inference(group, demo_groups=demo_groups_c, scheme_hint=COT_HINT)
        else:
            generated = run_inference(group, demo_groups=None, scheme_hint=None)
        pred_ranks = parse_ranking(generated, group)

    else:
        # TOPSIS or LambdaMART — deterministic, no GPU needed
        pred_ranks = get_predictions(group)

    m = compute_metrics_df(group, pred_ranks)
    m['list_id']    = lid
    m['phase']      = phase
    m['list_index'] = idx
    results.append(m)
    done_ids.add(unique_key)

    # Save checkpoint every 10 lists
    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

# Final save
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_FILE, index=False)

phase_b = df_out[df_out['phase'] == 'B']
phase_c = df_out[df_out['phase'] == 'C']
print(f'\nDone — {ORCHESTRATOR}')
print(f'  Phase B Top-1: {phase_b["top1"].mean()*100:.2f}%')
print(f'  Phase C Top-1: {phase_c["top1"].mean()*100:.2f}%')
print(f'  Saved: {OUTPUT_FILE}')

Running orchestrator: qwen_promptadapt
Total lists to run:   502
Phase boundary at:    list index 251
  [0/502] Phase B, list_id 0...
  [25/502] Phase B, list_id 25...
  [50/502] Phase B, list_id 50...
  [75/502] Phase B, list_id 75...
  [100/502] Phase B, list_id 100...
  [125/502] Phase B, list_id 125...
  [150/502] Phase B, list_id 150...
  [175/502] Phase B, list_id 175...
  [200/502] Phase B, list_id 200...
  [225/502] Phase B, list_id 225...
  [250/502] Phase B, list_id 250...
  [275/502] Phase C, list_id 24...
  [300/502] Phase C, list_id 49...
  [325/502] Phase C, list_id 74...
  [350/502] Phase C, list_id 99...
  [375/502] Phase C, list_id 124...
  [400/502] Phase C, list_id 149...
  [425/502] Phase C, list_id 174...
  [450/502] Phase C, list_id 199...
  [475/502] Phase C, list_id 224...
  [500/502] Phase C, list_id 249...

Done — qwen_promptadapt
  Phase B Top-1: 90.44%
  Phase C Top-1: 63.75%
  Saved: /content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_Dynamic_Adapt

In [9]:
# CELL 8: Run the selected orchestrator across all 502 lists
# Checkpoint-enabled: safe to interrupt and resume
# IMPORTANT: done_ids uses 'phase_listid' key to avoid collision
# (Phase B and Phase C both have list_ids 0-250)

import torch

OUTPUT_FILE = RESULTS_PATH + f'{ORCHESTRATOR}_502.csv'

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_ids = set((existing['phase'] + '_' + existing['list_id'].astype(str)).tolist())
    results  = existing.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_ids)} lists done')
else:
    done_ids = set()
    results  = []

# Build full trace: Phase 1 = Scheme B, Phase 2 = Scheme C
trace = (
    [(lid, test_b, 'B', i)                   for i, lid in enumerate(list_ids_b)] +
    [(lid, test_c, 'C', len(list_ids_b) + i) for i, lid in enumerate(EVAL_LIST_IDS_C)]
)

print(f'Running orchestrator: {ORCHESTRATOR}')
print(f'Total lists to run:   {len(trace)}')
print(f'Phase boundary at:    list index {len(list_ids_b)}')

COT_HINT = (
    "New ranking rule: sort nodes by Availability descending. "
    "Use Reliability as tiebreaker. "
    "Ignore all other metrics. "
    "Now rank:"

)

for list_index, (lid, source_df, phase, idx) in enumerate(trace):
    unique_key = f'{phase}_{lid}'
    if unique_key in done_ids:
        continue

    if list_index % 25 == 0:
        print(f'  [{list_index}/{len(trace)}] Phase {phase}, list_id {lid}...')

    group = source_df[source_df['list_id'] == lid].copy()
    group = group.sample(frac=1, random_state=lid).reset_index(drop=True)

    # Get predictions based on orchestrator type
    if ORCHESTRATOR not in ('topsis_stale', 'lambdamart_b'):
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        if phase == 'C' and 'promptadapt' in ORCHESTRATOR:
            generated = run_inference(group, demo_groups=demo_groups_c, scheme_hint=COT_HINT)
        else:
            generated = run_inference(group, demo_groups=None, scheme_hint=None)
        pred_ranks = parse_ranking(generated, group)

    else:
        # TOPSIS or LambdaMART — deterministic, no GPU needed
        pred_ranks = get_predictions(group)

    m = compute_metrics_df(group, pred_ranks)
    m['list_id']    = lid
    m['phase']      = phase
    m['list_index'] = idx
    results.append(m)
    done_ids.add(unique_key)

    # Save checkpoint every 10 lists
    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

# Final save
df_out = pd.DataFrame(results)
df_out.to_csv(OUTPUT_FILE, index=False)

phase_b = df_out[df_out['phase'] == 'B']
phase_c = df_out[df_out['phase'] == 'C']
print(f'\nDone — {ORCHESTRATOR}')
print(f'  Phase B Top-1: {phase_b["top1"].mean()*100:.2f}%')
print(f'  Phase C Top-1: {phase_c["top1"].mean()*100:.2f}%')
print(f'  Saved: {OUTPUT_FILE}')

Running orchestrator: qwen_noadapt
Total lists to run:   502
Phase boundary at:    list index 251
  [0/502] Phase B, list_id 0...
  [25/502] Phase B, list_id 25...
  [50/502] Phase B, list_id 50...
  [75/502] Phase B, list_id 75...
  [100/502] Phase B, list_id 100...
  [125/502] Phase B, list_id 125...
  [150/502] Phase B, list_id 150...
  [175/502] Phase B, list_id 175...
  [200/502] Phase B, list_id 200...
  [225/502] Phase B, list_id 225...
  [250/502] Phase B, list_id 250...
  [275/502] Phase C, list_id 24...
  [300/502] Phase C, list_id 49...
  [325/502] Phase C, list_id 74...
  [350/502] Phase C, list_id 99...
  [375/502] Phase C, list_id 124...
  [400/502] Phase C, list_id 149...
  [425/502] Phase C, list_id 174...
  [450/502] Phase C, list_id 199...
  [475/502] Phase C, list_id 224...
  [500/502] Phase C, list_id 249...

Done — qwen_noadapt
  Phase B Top-1: 90.44%
  Phase C Top-1: 53.39%
  Saved: /content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_Dynamic_Adaptation/qw

In [ ]:
# CELL 9: Plot and recovery analysis
# Run this after ALL orchestrators are complete — loads all available CSV files automatically

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

WINDOW = 20

def rolling_top1(df, window=WINDOW):
    return df.sort_values('list_index').reset_index(drop=True)['top1'].rolling(window, min_periods=1).mean() * 100

# Auto-discover all orchestrator result files
all_files = [f for f in os.listdir(RESULTS_PATH) if f.endswith('_502.csv')]
all_files.sort()

color_pool     = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'gray']
linestyle_pool = ['-', '-', '-', '-', '--', '--', '--', '--']

dfs = {}
for fname in all_files:
    label = fname.replace('_502.csv', '').replace('_', ' ')
    path  = RESULTS_PATH + fname
    dfs[label] = pd.read_csv(path).sort_values('list_index')
    print(f'Loaded: {fname}')

fig, ax = plt.subplots(figsize=(12, 5))
for (label, df), color, ls in zip(dfs.items(), color_pool, linestyle_pool):
    ax.plot(df['list_index'], rolling_top1(df), label=label, color=color, linewidth=1.8, linestyle=ls)

boundary = 251
ax.axvline(x=boundary, color='black', linestyle=':', linewidth=1.5, label='SLA shift (B→C)')
ax.axvspan(0, boundary, alpha=0.05, color='blue')
ax.axvspan(boundary, boundary + 251, alpha=0.05, color='orange')
ax.text(boundary/2, 5, 'Phase 1 (Scheme B)', ha='center', fontsize=9, color='navy')
ax.text(boundary + 125, 5, 'Phase 2 (Scheme C)', ha='center', fontsize=9, color='darkorange')

ax.set_xlabel('List Index (request sequence)', fontsize=11)
ax.set_ylabel(f'Top-1 Accuracy (%) — rolling window {WINDOW}', fontsize=11)
ax.set_title('Dynamic In-Context Adaptation Under Shifting SLA Priorities', fontsize=12)
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim([0, 105])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_PATH + 'dynamic_adaptation_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

# Recovery analysis — find best prompt-adapt result for target
prompt_adapt_files = [f for f in all_files if 'promptadapt' in f]
if prompt_adapt_files:
    best_c = 0
    for fname in prompt_adapt_files:
        df_tmp = pd.read_csv(RESULTS_PATH + fname)
        c_acc  = df_tmp[df_tmp['phase'] == 'C']['top1'].mean() * 100
        if c_acc > best_c:
            best_c = c_acc
    target = best_c - 2
    print(f'\nRecovery target (within 2pp of best prompt-adapt Phase C={best_c:.2f}%): >= {target:.2f}%')
else:
    target = 60.0
    print(f'\nNo prompt-adapt results found. Using default target: >= {target:.2f}%')

print('='*60)
for label, df in dfs.items():
    b_top1 = df[df['phase']=='B']['top1'].mean()*100
    c_top1 = df[df['phase']=='C']['top1'].mean()*100
    df_c   = df[df['phase']=='C'].sort_values('list_index').reset_index(drop=True)
    rolling = df_c['top1'].rolling(20, min_periods=20).mean() * 100
    recovered = None
    for i, v in enumerate(rolling):
        if pd.notna(v) and v >= target:
            recovered = i
            break
    rec_str = f'{recovered} requests' if recovered is not None else 'never recovered'
    print(f'{label}:  Phase B={b_top1:.2f}%  Phase C={c_top1:.2f}%  Recovery={rec_str}')

Loaded: lambdamart_b_502.csv
Loaded: qwen_noadapt_502.csv
Loaded: qwen_promptadapt_502.csv
Loaded: topsis_stale_502.csv
Plot saved.

Recovery target (within 2pp of best prompt-adapt Phase C=63.75%): >= 61.75%
lambdamart b:  Phase B=91.63%  Phase C=52.59%  Recovery=140 requests
qwen noadapt:  Phase B=90.44%  Phase C=53.39%  Recovery=63 requests
qwen promptadapt:  Phase B=90.44%  Phase C=63.75%  Recovery=19 requests
topsis stale:  Phase B=92.03%  Phase C=55.38%  Recovery=140 requests
